# Paired-patch controls & component-attribution stability

Hardens the strongest mechanistic result (success->failure paired patching) and shows the component
story is stable, not noisy top-5 archaeology.
- **PC1 Four-arm paired-patch control**: for each failed paraphrase, patch late components from
  (a) the SAME fact's success, (b) a DIFFERENT fact's success, (c) RANDOM late components of the same
  success, (d) EARLY components of the same success. Report margin shift, first-token recovery, and
  answer/alternative support change. If (a) >> (b),(c),(d), the effect is not "any late activation helps."
- **PC2 Attribution stability**: bootstrap the component attribution -- top-k overlap, rank correlation,
  late-layer mass, attention-vs-MLP mass.
- **PC3 Mean-residual / baseline ablation** (EXPLORATORY): subtract a fraction of the mean residual or
  remove only its frequency-direction projection; measure frequency shift / margin / recovery.

Needs `rw_core.py`, `mech_core.py`, `mech_runner.py`, `component_core.py`, `baseline_core.py`.

## 0. Config + data

In [ ]:
import numpy as np, json, gc, torch, re
import rw_core as rw, mech_core as mc, mech_runner as mr, component_core as cc, baseline_core as bc
from datasets import load_dataset
from collections import defaultdict
import pandas as pd

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32
QA="Answer with a short factual answer.\nQuestion: {q}\nAnswer:"
TEMPLATES=[QA,"Q: {q}\nA:","Please answer concisely.\n{q}\nAnswer:","{q} The answer is"]
N_ITEMS=300; CAPS=dict(ctrl=80, stab=200, abl=200)

MODELS=[
 "meta-llama/Llama-3.1-8B","meta-llama/Llama-3.2-3B","meta-llama/Llama-3.2-1B",
 "meta-llama/Llama-3.2-3B-Instruct","Qwen/Qwen2.5-3B","Qwen/Qwen2.5-3B-Instruct",
 "Qwen/Qwen2.5-7B","mistralai/Mistral-7B-v0.1",
]
# component capture is heavy; start with MODELS[:2] to time it.

ds=load_dataset("akariasai/PopQA",split="test")
def aliases(r):
    a=r["possible_answers"]
    if isinstance(a,str):
        try: a=json.loads(a)
        except: a=[a]
    return a
ITEMS=[{"q":str(r["question"]),"gold":aliases(r),"rel":str(r.get("prop","na"))} for r in ds]
REL=defaultdict(list)
for it in ITEMS:
    for a in it["gold"]: REL[it["rel"]].append(a)
np.random.default_rng(0).shuffle(ITEMS); ITEMS=ITEMS[:N_ITEMS]
print("items per model:",len(ITEMS))

## 1. Run PC1/PC2/PC3 on all models (per-experiment isolation)

In [ ]:
PATCH={}
for name in MODELS:
    print("="*70); print(name,flush=True)
    try:
        ctx=mr.make_ctx(name,DEVICE,DTYPE,ITEMS,REL,QA,TEMPLATES)
        out={}
        for key,fnc in [
            ("PC1_ctrl", lambda: mr.exp_paired_patch_controls(ctx, max_facts=CAPS["ctrl"])),
            ("PC2_stab", lambda: mr.exp_component_stability(ctx, max_items=CAPS["stab"], n_boot=30, top_k=8)),
            ("PC3_abl",  lambda: mr.exp_mean_residual_ablation(ctx, ctx["freq"], max_items=CAPS["abl"])),
        ]:
            try: out[key]=fnc(); print(f"  {key} done",flush=True)
            except Exception as e:
                import traceback; traceback.print_exc(); out[key]={"status":"error","error":f"{type(e).__name__}: {e}"}
                print(f"  {key} FAILED: {type(e).__name__}: {str(e)[:140]}",flush=True)
        PATCH[name]=out
    except Exception as e:
        import traceback; traceback.print_exc(); PATCH[name]={"status":"error","error":f"{type(e).__name__}: {e}"}
    finally:
        try: mr.free_ctx(ctx); del ctx
        except Exception: pass
        torch.cuda.empty_cache(); gc.collect()
json.dump(PATCH,open("paired_patch_controls.json","w"),indent=2,default=float)
print("\nsucceeded:",len([k for k,v in PATCH.items() if "status" not in v]),"/",len(PATCH))

## 2. PC1 — four-arm paired-patch control (the key hardening table)
same_fact = real effect; diff_fact / rand_late / early = controls. The result holds if same_fact has
the largest margin shift and recovery, and the controls are near zero.

In [ ]:
for arm_metric in ["mean_shift","recovery","d_answer_support","d_alt_support"]:
    print(f"\n=== {arm_metric} ===")
    rows=[]
    for nm,v in PATCH.items():
        if "status" in v or "status" in v.get("PC1_ctrl",{}): continue
        a=v["PC1_ctrl"]
        if "same_fact" not in a: continue
        rows.append({"model":nm.split("/")[-1],
                     "same_fact":round(a["same_fact"][arm_metric],3),
                     "diff_fact":round(a["diff_fact"][arm_metric],3),
                     "rand_late":round(a["rand_late"][arm_metric],3),
                     "early":round(a["early"][arm_metric],3)})
    print(pd.DataFrame(rows).to_string(index=False))
print("\nsame_fact >> diff_fact, rand_late, early on margin_shift/recovery = the patch effect is")
print("specific to the same fact's late components, not 'any late activation helps.'")

## 3. PC2 — component attribution stability

In [ ]:
rows=[{"model":nm.split("/")[-1],"n":v["PC2_stab"]["n"],
        "topk_overlap":round(v["PC2_stab"]["mean_topk_overlap"],3),
        "min_overlap":round(v["PC2_stab"]["min_topk_overlap"],3),
        "rank_corr":round(v["PC2_stab"]["mean_rank_corr"],3),
        "late_mass":round(v["PC2_stab"]["late_mass_frac"],3),
        "attn_mass":round(v["PC2_stab"]["attn_mass_frac"],3),
        "mlp_mass":round(v["PC2_stab"]["mlp_mass_frac"],3)}
       for nm,v in PATCH.items() if "status" not in v and "status" not in v.get("PC2_stab",{})]
print(pd.DataFrame(rows).to_string(index=False))
print("\nhigh topk_overlap + rank_corr across bootstraps = the component story is stable. late_mass and")
print("attn vs mlp mass say where the alternative-favoring contribution concentrates.")
for nm,v in PATCH.items():
    if "status" in v or "status" in v.get("PC2_stab",{}): continue
    print(f"  {nm.split('/')[-1]} top components:", ", ".join(c.replace('attn.','a').replace('mlp.','m') for c in v["PC2_stab"]["full_top_components"]))

## 4. PC3 — mean-residual / baseline ablation (EXPLORATORY)
Subtracting the mean residual can break the representation broadly; read as exploratory. Removing only
the frequency-direction projection is the gentler, more interpretable variant.

In [ ]:
for nm,v in PATCH.items():
    if "status" in v or "status" in v.get("PC3_abl",{}): continue
    a=v["PC3_abl"]
    if "status" in a: print(nm,"->",a.get("status")); continue
    print("###",nm.split("/")[-1],f"(n={a['n']})")
    for k in ["subtract_meanres_alpha0.5","subtract_meanres_alpha1.0","remove_freqdir_projection","remove_meanres_projection"]:
        if k in a:
            d=a[k]; print(f"   {k:32s}: margin_shift={d['margin_shift']:+.3f} recovery={d['recovery']:.3f} sel_freq_shift={d['mean_selected_freq_shift']:+.3f}")
print("\nremove_freqdir_projection demoting the selected token's frequency (negative sel_freq_shift)")
print("with a positive margin_shift would support the baseline being frequency-graded -- exploratory.")

## Notes
- **PC1 is the hardening of the cleanest causal result.** Report all four arms; same_fact dominating
  margin_shift and recovery while diff_fact/rand_late/early stay near zero is the answer to "any late
  activation helps." Report d_answer_support and d_alt_support too -- same_fact should raise answer
  support and/or lower alternative support more than the controls.
- **PC2** answers "is this noisy top-5 archaeology": high bootstrap top-k overlap and rank correlation
  say no. late_mass / attn_mass / mlp_mass connect to the head-localization results.
- **PC3** is exploratory and may be messy (subtracting the mean residual is a broad edit); lead with
  remove_freqdir_projection, the gentler variant. Don't over-claim from it.
- Component capture is heavy; start with `MODELS[:2]`, raise `CAPS` for finals.
- I would NOT add further semantic / WordNet / frequency-regression / training-dynamics experiments
  unless a specific reviewer asks; these controls are the high-value remaining work.